In [1]:
import io
import os
import pandas as pd

from contextlib import redirect_stdout
from dotenv import load_dotenv

In [2]:
load_dotenv(r'../.env')

True

## FRED API - Economic data

In [3]:
import pyfredapi as pf
from pyfredapi._base import FredAPIRequestError 

API_KEY = os.getenv('FRED_API_KEY')

### Configuration

In [4]:
# raw data output directory
OUT_DIR = r"C:\Users\mushj\Downloads\RAW FINANCE DATA\FRED\RL_PROJECT"
VERSION = '002'

FRED_PARAMS = {
    'observation_start': '1975-01-01',
    'observation_end': '2025-12-05'
}

SERIES_ID_LIST = [
    'VIXCLS',
    'DFF',
    'T10Y2Y',
    'REAINTRATREARAT10Y',
    'BAMLH0A0HYM2',
    'CPIAUCSL',
    'ICSA',
    'PPIACO',
    'USSTHPI',
]

### Collect data

In [ ]:
# store printed string
buffer = io.StringIO()

# download data for each series
with redirect_stdout(buffer):
    for series_id in SERIES_ID_LIST:
        # get series info
        try:
            series_info = pf.get_series_info(series_id=series_id, api_key=API_KEY)
        except FredAPIRequestError as e:
            msg = str(e)
            if 'The series does not exist' in msg:
                print(f'The series "{series_id}" does not exist')
            else:
                print(msg)
            continue

        # print key series info.
        print(f'id: {series_id} | title: {series_info.title}')
        print(f'obs range: {series_info.observation_start} to {series_info.observation_end}')
        print(f'freq: {series_info.frequency} | units: {series_info.units}')
        print(series_info.seasonal_adjustment)

        # get data
        df = pf.get_series(series_id=series_id, api_key=API_KEY, **FRED_PARAMS)
        print(df.shape, '\n')

        # export data to local storage
        # skips if already exists (prevents overwrite)
        file_name = f'{series_id}_{VERSION}.csv'
        out_path = os.path.join(OUT_DIR, file_name)
        try:
            df.to_csv(out_path, index=False, mode='x')
        except FileExistsError:
            print(f'{file_name} already exists in the output directory.')

# write metadata
metadata_file = os.path.join(OUT_DIR, f"metadata_{VERSION}.txt")
with open(metadata_file, "w") as f:
    f.write(buffer.getvalue())

## yfinance - Stock prices

In [6]:
import yfinance as yf

### Configuration

In [7]:
# raw data output directory
YF_OUT_DIR = r'C:\Users\mushj\Downloads\RAW FINANCE DATA\YFINANCE\RL_PROJECT'
YF_VERSION = '001'

YF_PARAMS = {
    'period': 'max',
    'interval': '1d',
    'start': '1975-01-01',
    'end': '2025-12-05',
    'keepna': True,
    'actions': False,
    'multi_level_index': True
}

TICKER_LIST = [
    'SPY', 
    'AAPL', 
    'NVDA', 
    'MSFT', 
    'AMZN', 
    'GOOG', 
    'JPM', 
    'XOM',
    'PG', 
    'UNH'
]

### Collect data

In [8]:
df = yf.download(
    tickers=TICKER_LIST,
    **YF_PARAMS
)

df.shape

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  10 of 10 completed


(12840, 50)

In [9]:
# export data to local storage
# skips if already exists (prevents overwrite)
file_name = f"prices_{YF_VERSION}.csv"
out_path = os.path.join(YF_OUT_DIR, file_name)
try:
    df.reset_index().to_csv(out_path, index=False, mode='x')
except FileExistsError:
    print(f"{file_name} already exists in the output directory.")

prices_001.csv already exists in the output directory.
